In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/cab_rides_with_weather.csv", parse_dates=["ride_time", "weather_time"])
df.shape

(637976, 16)

In [2]:
# raw price cannot be compared across rides of different lengths, dividing by distance puts every ride on the same footing so routes and tiers become comparable
df["price_per_mile"] = df["price"] / df["distance"]
df["price_per_mile"].describe()

count    637976.000000
mean          9.687674
std          13.705589
min           0.556586
25%           4.661017
50%           7.492326
75%          11.538462
max        1375.000000
Name: price_per_mile, dtype: float64

In [3]:
df["ride_hour"] = df["ride_time"].dt.hour
df["ride_day_of_week"] = df["ride_time"].dt.dayofweek
df["is_weekend"] = df["ride_day_of_week"].isin([5, 6])
df[["ride_hour", "ride_day_of_week", "is_weekend"]].head()

,ride_hour,ride_day_of_week,is_weekend
0,3,0,False
1,3,0,False
2,3,0,False
3,3,0,False
4,3,0,False


In [4]:
# grouping the raw hour into named parts of the day gives the model a coarser, more interpretable time signal than the hour number alone
bins = [-1, 5, 11, 16, 20, 23]
labels = ["late_night", "morning", "afternoon", "evening", "night"]
df["time_of_day"] = pd.cut(df["ride_hour"], bins=bins, labels=labels)
df["time_of_day"].value_counts()

time_of_day
late_night    157155
morning       152951
afternoon     139810
evening       106063
night          81997
Name: count, dtype: int64

In [5]:
# surge_multiplier is real for Lyft but is always 1.0 for Uber, Uber never exposed its live multiplier through this API and folded the markup straight into the price instead, so this column cannot be used as a surge signal across both platforms
df.groupby("cab_type")["surge_multiplier"].value_counts()

cab_type  surge_multiplier
Lyft      1.00                286433
          1.25                 11085
          1.50                  5065
          1.75                  2420
          2.00                  2239
          2.50                   154
          3.00                    12
Uber      1.00                330568
Name: count, dtype: int64

In [6]:
# instead we build our own surge signal that works for both platforms: how much a ride's price-per-mile sits above the normal, non-surged rate for that exact route and service tier, since price-per-mile already varies a lot tier to tier and route to route, the comparison has to be made within each (source, destination, name) group rather than across the whole dataset
group_cols = ["source", "destination", "name"]
group_sizes = df.groupby(group_cols).size()
print("number of route+tier groups:", group_sizes.shape[0])
print("smallest group size:", group_sizes.min())

number of route+tier groups: 864
smallest group size: 672


In [7]:
# every route+tier group has well over 600 rides, so a percentile computed within a group will be stable and not driven by one or two stray fares, no fallback to a coarser grouping is needed here
MIN_GROUP_SIZE = 30
print("groups below the minimum reliable size:", (group_sizes < MIN_GROUP_SIZE).sum())

groups below the minimum reliable size: 0


In [8]:
# for Lyft we know exactly which rides were not surged, so the base rate is taken only from those rides, using the 10th percentile rather than the mean or median because fares only ever move up from the base rate during surge, never down, so the low end of the distribution is the best estimate of the true floor
lyft_base_rows = df[(df["cab_type"] == "Lyft") & (df["surge_multiplier"] == 1.0)]
lyft_base_price = lyft_base_rows.groupby(group_cols)["price_per_mile"].quantile(0.10)
lyft_base_price.name = "base_price_per_mile"
lyft_base_price.head()

source    destination        name        
Back Bay  Boston University  Lux              8.823529
                             Lux Black       10.784314
                             Lux Black XL    17.105263
                             Lyft             4.575163
                             Lyft XL          7.175537
Name: base_price_per_mile, dtype: float64

In [9]:
# Uber has no surge flag to filter on, so the floor is taken from all Uber rides in the group instead, this is noisier than the Lyft floor but it is the only information available
uber_rows = df[df["cab_type"] == "Uber"]
uber_base_price = uber_rows.groupby(group_cols)["price_per_mile"].quantile(0.10)
uber_base_price.name = "base_price_per_mile"
uber_base_price.head()

source    destination        name     
Back Bay  Boston University  Black        10.638298
                             Black SUV    15.972222
                             UberPool      4.609929
                             UberX         5.298013
                             UberXL        8.156028
Name: base_price_per_mile, dtype: float64

In [10]:
base_price_lookup = pd.concat([lyft_base_price, uber_base_price]).reset_index()
df = df.merge(base_price_lookup, on=group_cols, how="left")
df["base_price_per_mile"].isnull().sum()

np.int64(0)

In [11]:
df["effective_surge"] = df["price_per_mile"] / df["base_price_per_mile"]
df["effective_surge"].describe()

count    637976.000000
mean          1.228583
std           1.005037
min           0.224138
25%           1.031746
50%           1.119105
75%           1.257286
max          75.000000
Name: effective_surge, dtype: float64

In [12]:
# the max of 75x is not real surge, it comes from a handful of rides with distance as low as 0.02 miles where dividing the price by such a tiny distance blows up price-per-mile, looking at the worst offenders confirms this
df.sort_values("effective_surge", ascending=False)[["distance", "price", "price_per_mile", "base_price_per_mile", "effective_surge"]].head(10)

,distance,price,price_per_mile,base_price_per_mile,effective_surge
460437,0.02,7.5,375.0,5.000000,75.000000
390984,0.02,7.5,375.0,5.000000,75.000000
136870,0.02,7.5,375.0,5.384615,69.642857
64376,0.02,7.5,375.0,5.384615,69.642857
64373,0.02,7.5,375.0,5.384615,69.642857
387083,0.02,7.5,375.0,5.384615,69.642857
55531,0.02,27.5,1375.0,20.000000,68.750000
64375,0.02,27.5,1375.0,20.000000,68.750000
8377,0.02,27.5,1375.0,20.000000,68.750000
192112,0.02,27.5,1375.0,20.000000,68.750000


In [13]:
# the real surge_multiplier column never goes above 3.0, so anything past that in our proxy is a short-distance artifact rather than genuine demand pricing, capping at 5.0 keeps some headroom above the real maximum while removing the artifacts
print("rides affected by the cap:", (df["effective_surge"] > 5.0).sum())
df["effective_surge"] = df["effective_surge"].clip(upper=5.0)
df["effective_surge"].describe()

rides affected by the cap: 520


count    637976.000000
mean          1.208883
std           0.305344
min           0.224138
25%           1.031746
50%           1.119105
75%           1.257286
max           5.000000
Name: effective_surge, dtype: float64

In [14]:
# sanity check against ground truth: for Lyft we have the real surge_multiplier, if our proxy is working its mean within each true surge level should land close to that level
df[df["cab_type"] == "Lyft"].groupby("surge_multiplier")["effective_surge"].agg(["mean", "median", "count"])

,mean,median,count
surge_multiplier,,,
1.00,1.208664,1.114679,286433
1.25,1.376653,1.355769,11085
1.50,1.624572,1.596629,5065
1.75,1.886608,1.860003,2420
2.00,2.119878,2.064068,2239
2.50,2.525418,2.490704,154
3.00,3.008945,3.015051,12


In [15]:
# there is no rider or driver count in this data, so as a demand proxy we use how many rides were requested at the same pickup location within the same hour, a busier hour at a location is a reasonable stand-in for higher demand
df["ride_hour_bucket"] = df["ride_time"].dt.floor("h")
demand_counts = df.groupby(["source", "ride_hour_bucket"]).size()
demand_counts.name = "rides_in_hour_at_source"
demand_counts.describe()

count    3984.000000
mean      160.134538
std        66.566064
min         1.000000
25%       129.000000
50%       145.000000
75%       167.000000
max       485.000000
Name: rides_in_hour_at_source, dtype: float64

In [16]:
df = df.merge(demand_counts.reset_index(), on=["source", "ride_hour_bucket"], how="left")
df["rides_in_hour_at_source"].isnull().sum()

np.int64(0)

In [17]:
# turning the raw count into quartile buckets gives the model a simple, interpretable demand_density category instead of a raw count whose scale only makes sense relative to the rest of the data
df["demand_density"] = pd.qcut(df["rides_in_hour_at_source"], q=4, labels=["low", "medium", "high", "very_high"])
df["demand_density"].value_counts()

demand_density
low          162224
high         159880
very_high    158290
medium       157582
Name: count, dtype: int64

In [18]:
# Uber and Lyft compete on the same routes, so how one platform's average price compares to the other's on the same route in the same hour captures competitive pricing pressure that neither platform's own data shows on its own. Using price_per_mile rather than raw price, since the same route+hour window can have a different mix of trip lengths between the two platforms, and that mix shouldn't be mistaken for competitive pricing pressure. Using the median rather than the mean for the same reason the surge proxy avoided the mean earlier: a single near-zero-distance ride blows up price_per_mile, and the median is far less sensitive to that one outlier dragging the whole group's value with it.
route_hour_avg = df.groupby(["source", "destination", "ride_hour_bucket", "cab_type"])["price_per_mile"].median().reset_index()
price_pivot = route_hour_avg.pivot_table(index=["source", "destination", "ride_hour_bucket"], columns="cab_type", values="price_per_mile").reset_index()
price_pivot["uber_lyft_price_ratio"] = price_pivot["Uber"] / price_pivot["Lyft"]
price_pivot["uber_lyft_price_ratio"].describe()

count    23338.000000
mean         0.887132
std          0.537750
min          0.107774
25%          0.664903
50%          0.830242
75%          1.015473
max         56.256656
Name: uber_lyft_price_ratio, dtype: float64

In [19]:
# a small number of route+hour combinations only had one of the two platforms operating, those rides have no competitor price to compare against, leaving the ratio as missing is more honest than inventing a value for a competitor that was not actually there
df = df.merge(price_pivot[["source", "destination", "ride_hour_bucket", "uber_lyft_price_ratio"]], on=["source", "destination", "ride_hour_bucket"], how="left")
df["uber_lyft_price_ratio"].isnull().sum()

np.int64(3071)

In [20]:
# rain is logged as an intensity, not a yes or no, but a simple boolean for whether it is raining at all is an easier signal for a model to pick up on alongside the continuous rain amount
df["is_raining"] = df["rain"] > 0
df["is_raining"].value_counts()

is_raining
False    544429
True      93547
Name: count, dtype: int64

In [21]:
# clouds is already a 0 to 1 fraction, rain and wind are on their own scales, so each is divided by its own max to bring it onto the same 0 to 1 range before they can be averaged into a single weather severity score
rain_normalized = df["rain"] / df["rain"].max()
wind_normalized = df["wind"] / df["wind"].max()
df["bad_weather_score"] = (df["clouds"] + rain_normalized + wind_normalized) / 3
df["bad_weather_score"].describe()

count    637976.000000
mean          0.353597
std           0.136391
min           0.042721
25%           0.250678
50%           0.368915
75%           0.448662
max           0.904587
Name: bad_weather_score, dtype: float64

In [22]:
df.shape

(637976, 29)

In [23]:
df.isnull().sum()

distance                      0
cab_type                      0
destination                   0
source                        0
price                         0
surge_multiplier              0
name                          0
ride_time                     0
temp                          0
clouds                        0
pressure                      0
rain                          0
humidity                      0
wind                          0
weather_time                  0
time_gap_minutes              0
price_per_mile                0
ride_hour                     0
ride_day_of_week              0
is_weekend                    0
time_of_day                   0
base_price_per_mile           0
effective_surge               0
ride_hour_bucket              0
rides_in_hour_at_source       0
demand_density                0
uber_lyft_price_ratio      3071
is_raining                    0
bad_weather_score             0
dtype: int64

In [24]:
df.head()

,distance,cab_type,destination,source,price,surge_multiplier,name,ride_time,temp,clouds,...,is_weekend,time_of_day,base_price_per_mile,effective_surge,ride_hour_bucket,rides_in_hour_at_source,demand_density,uber_lyft_price_ratio,is_raining,bad_weather_score
0,3.03,Lyft,Theatre District,Boston University,34.0,1.0,Lux Black XL,2018-11-26 03:40:46.318,41.07,0.86,...,False,late_night,10.833333,1.035796,2018-11-26 03:00:00,9,low,NaN,False,0.311602
1,1.30,Uber,Theatre District,South Station,18.5,1.0,Black,2018-11-26 03:40:46.319,40.86,0.87,...,False,late_night,11.538462,1.233333,2018-11-26 03:00:00,11,low,NaN,False,0.319336
2,2.43,Lyft,Beacon Hill,Northeastern University,10.5,1.0,Lyft,2018-11-26 03:40:46.320,40.81,0.89,...,False,late_night,3.673469,1.176269,2018-11-26 03:00:00,4,low,NaN,False,0.321602
3,2.71,Uber,Fenway,Theatre District,32.0,1.0,UberXL,2018-11-26 03:40:46.320,40.80,0.87,...,False,late_night,5.639098,2.093973,2018-11-26 03:00:00,13,low,3.351081,False,0.318420
4,2.71,Uber,Fenway,Theatre District,19.5,1.0,UberX,2018-11-26 03:40:46.320,40.80,0.87,...,False,late_night,3.571429,2.014760,2018-11-26 03:00:00,13,low,3.351081,False,0.318420


In [25]:
df.to_csv("../data/cab_rides_features.csv", index=False)